# RFT-0004B — R2 Pro4-hint Qwen low-drift continuation LoRA

Continues the frozen Lane-B checkpoint-178 adapter on the audited R2
65% rationalized-hard / 35% untouched-R1-anchor corpus. It uses BF16,
assistant-only loss, LR 1e-5, 0.5 epoch, effective batch 48, and saves
quarter checkpoints. No leaderboard/test/Pro4 API is read or called.

In [ ]:
# Cell 1 — Tested training stack.
%pip install -q --no-cache-dir "transformers==4.52.3" "peft==0.16.0" "accelerate==1.7.0" \
  "datasets==3.6.0" "bitsandbytes==0.46.0" "sentencepiece>=0.2.0" \
  "tensorboard~=2.19.0" "wandb>=0.19,<1" "protobuf<6"
print("[SETUP] complete; restart once only if these packages were already imported")

In [ ]:
# Cell 2 — Paths, immutable inputs, and low-drift configuration.
import gc, hashlib, inspect, json, math, os, platform, re, time, unicodedata
from pathlib import Path
os.environ["TOKENIZERS_PARALLELISM"]="false";os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"
os.environ["WANDB_MODE"]="offline";os.environ["WANDB_PROJECT"]="deep-learning-challenge-2026"
import numpy as np, pandas as pd, torch
from google.colab import drive
BASE_MODEL="Qwen/Qwen2.5-3B-Instruct";MODEL_REVISION="aa8e72537993ba99e69dfaafa59ed015b17504d1";RUN_ID="RFT-0004B-r2-pro4-hint-lowdrift-lora";SEED=20260826
SOURCE_RUN_ID="RFT-0003A-pro4-hint-qwen-generation";EVAL_RUN_ID="RFT-0002-r1-ab-vllm-eval-a100"
MAX_SEQ_LENGTH=2048;MICRO_BATCH=8;GRAD_ACCUM=6;EFFECTIVE_BATCH=48;EPOCHS=.5;LEARNING_RATE=1e-5
SYSTEM_PROMPT="You are a helpful assistant that solves math problems step by step."
USER_SUFFIX="Solve this step by step, then give the final answer as a single integer inside \\boxed{}."
def compact(v):return re.sub(r"[\s_\-]+","",unicodedata.normalize("NFC",str(v)).casefold())
def sha(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for c in iter(lambda:f.read(1024*1024),b""):h.update(c)
    return h.hexdigest()
mount=Path("/content/drive")
if not (mount/"MyDrive").exists():drive.mount(str(mount))
roots=[p for p in (mount/"MyDrive").iterdir() if p.is_dir() and compact(p.name)==compact("2026소중한챌린지")];assert len(roots)==1
PROJECT_DIR=roots[0];RUNS_DIR=PROJECT_DIR/"runs"
DATA_PATH=RUNS_DIR/SOURCE_RUN_ID/"data"/"r2_pro4_hint_qwen_65_anchor35_train.csv"
DATA_REPORT=RUNS_DIR/SOURCE_RUN_ID/"reports"/"r2_data_generation_report.json"
winners=json.loads((RUNS_DIR/EVAL_RUN_ID/"reports"/"frozen_tune_winners.json").read_text())
PARENT_ADAPTER=Path(winners["lane_b_low_drift"]["path"]);assert PARENT_ADAPTER.name=="checkpoint-178"
parent_weight=next(p for p in [PARENT_ADAPTER/"adapter_model.safetensors",PARENT_ADAPTER/"adapter_model.bin"] if p.exists())
for p in [DATA_PATH,DATA_REPORT,parent_weight]:assert p.exists(),p
EXP_DIR=RUNS_DIR/RUN_ID;CHECKPOINT_DIR=EXP_DIR/"checkpoints";FINAL_ADAPTER=EXP_DIR/"adapter_final";REPORT_DIR=EXP_DIR/"reports";TB_DIR=EXP_DIR/"logs"/"tensorboard";WANDB_DIR=EXP_DIR/"logs"/"wandb"
for p in [CHECKPOINT_DIR,REPORT_DIR,TB_DIR,WANDB_DIR]:p.mkdir(parents=True,exist_ok=True)
os.environ["WANDB_DIR"]=str(WANDB_DIR)
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported();print("[GPU]",torch.cuda.get_device_name(0))
print("[DATA]",DATA_PATH,sha(DATA_PATH));print("[PARENT]",PARENT_ADAPTER,sha(parent_weight))

In [ ]:
# Cell 3 — Strict corpus verification and assistant-only tokenization.
from datasets import Dataset
from transformers import AutoTokenizer,set_seed
set_seed(SEED);tokenizer=AutoTokenizer.from_pretrained(BASE_MODEL,revision=MODEL_REVISION,use_fast=True,token=False)
tokenizer.pad_token=tokenizer.pad_token or tokenizer.eos_token;tokenizer.padding_side="right"
data=pd.read_csv(DATA_PATH,dtype=str,keep_default_na=False);data.columns=[c.strip() for c in data.columns]
assert {"id","question","answer","solution","source"}.issubset(data.columns);assert data["answer"].str.fullmatch(r"-?\d+").all();assert data.groupby("id").size().max()<=2
def prompt(q):return tokenizer.apply_chat_template([{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":f"{q.strip()}\n\n{USER_SUFFIX}"}],tokenize=False,add_generation_prompt=True)
def full(q,s):return tokenizer.apply_chat_template([{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":f"{q.strip()}\n\n{USER_SUFFIX}"},{"role":"assistant","content":s.strip()}],tokenize=False,add_generation_prompt=False)
def tok(e):
    p=tokenizer(prompt(e["question"]),add_special_tokens=False)["input_ids"];x=tokenizer(full(e["question"],e["solution"]),add_special_tokens=False)
    assert x["input_ids"][:len(p)]==p;labels=[-100]*len(p)+x["input_ids"][len(p):]
    return {"input_ids":x["input_ids"],"attention_mask":x["attention_mask"],"labels":labels,"total_tokens":len(x["input_ids"])}
ds=Dataset.from_pandas(data[["id","question","answer","solution","source"]],preserve_index=False).map(tok,num_proc=2,desc="Tokenize R2")
lengths=np.array(ds["total_tokens"]);keep=np.flatnonzero(lengths<=MAX_SEQ_LENGTH).tolist();reject=np.flatnonzero(lengths>MAX_SEQ_LENGTH).tolist()
assert len(reject)/len(ds)<=.02,(len(reject),len(ds));eligible=ds.select(keep)
ids=sorted(set(eligible["id"]),key=lambda x:hashlib.sha256(f"{SEED}|diag|{x}".encode()).hexdigest());eval_ids=set(ids[:min(256,max(1,len(ids)//20))])
ti=[i for i,x in enumerate(eligible["id"]) if x not in eval_ids];ei=[i for i,x in enumerate(eligible["id"]) if x in eval_ids]
remove=["id","question","answer","solution","source","total_tokens"]
train_dataset=eligible.select(ti).remove_columns(remove);eval_dataset=eligible.select(ei).remove_columns(remove)
pd.DataFrame({"id":sorted(eval_ids)}).to_csv(EXP_DIR/"diagnostic_eval_ids.csv",index=False)
print("[ROWS] source/eligible/train/eval/reject",len(data),len(eligible),len(train_dataset),len(eval_dataset),len(reject));print("[MAX TOKENS]",max(lengths[keep]))

In [ ]:
# Cell 4 — Load base and continue the exact parent Lane-B adapter trainably.
from transformers import AutoModelForCausalLM,DataCollatorForSeq2Seq
from peft import PeftModel
base=AutoModelForCausalLM.from_pretrained(BASE_MODEL,revision=MODEL_REVISION,torch_dtype=torch.bfloat16,device_map={"":0},token=False)
model=PeftModel.from_pretrained(base,str(PARENT_ADAPTER),is_trainable=True,adapter_name="default")
model.config.use_cache=False;model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant":False});model.enable_input_require_grads();model.train()
trainable=sum(p.numel() for p in model.parameters() if p.requires_grad);total=sum(p.numel() for p in model.parameters())
assert 0<trainable/total<.02;(model.print_trainable_parameters())
collator=DataCollatorForSeq2Seq(tokenizer=tokenizer,model=model,padding=True,label_pad_token_id=-100,pad_to_multiple_of=8)

In [ ]:
# Cell 5 — Low-drift half-epoch continuation with quarter checkpoints.
from transformers import Trainer,TrainingArguments
from transformers.trainer_utils import get_last_checkpoint
steps=max(1,math.ceil(len(train_dataset)/(MICRO_BATCH*GRAD_ACCUM)*EPOCHS));quarter=max(1,round(steps/4))
eval_arg="eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments).parameters else "evaluation_strategy"
kwargs={"output_dir":str(CHECKPOINT_DIR),"logging_dir":str(TB_DIR),"num_train_epochs":EPOCHS,"learning_rate":LEARNING_RATE,
        "per_device_train_batch_size":MICRO_BATCH,"per_device_eval_batch_size":4,"gradient_accumulation_steps":GRAD_ACCUM,
        "gradient_checkpointing":True,"gradient_checkpointing_kwargs":{"use_reentrant":False},"optim":"paged_adamw_8bit",
        "lr_scheduler_type":"cosine","warmup_ratio":.03,"weight_decay":0.,"max_grad_norm":1.,"bf16":True,"fp16":False,"tf32":True,
        "logging_steps":5,"logging_first_step":True,"eval_steps":quarter,"save_steps":quarter,"save_strategy":"steps","save_total_limit":5,
        "load_best_model_at_end":False,"prediction_loss_only":True,"report_to":["tensorboard","wandb"],"run_name":RUN_ID,
        "remove_unused_columns":False,"group_by_length":True,"dataloader_num_workers":2,"seed":SEED,"data_seed":SEED}
kwargs[eval_arg]="steps";trainer=Trainer(model=model,args=TrainingArguments(**kwargs),train_dataset=train_dataset,eval_dataset=eval_dataset,data_collator=collator)
resume=get_last_checkpoint(str(CHECKPOINT_DIR));print("[TRAIN] resume",resume,"steps",steps,"quarter",quarter)
started=time.time();result=trainer.train(resume_from_checkpoint=resume);runtime=time.time()-started
trainer.save_model(str(FINAL_ADAPTER));tokenizer.save_pretrained(str(FINAL_ADAPTER))
report={"run_id":RUN_ID,"status":"completed","parent_adapter":str(PARENT_ADAPTER),"parent_sha256":sha(parent_weight),"data":str(DATA_PATH),"data_sha256":sha(DATA_PATH),
        "train_rows":len(train_dataset),"eval_rows":len(eval_dataset),"epochs":EPOCHS,"learning_rate":LEARNING_RATE,"micro_batch":MICRO_BATCH,
        "gradient_accumulation":GRAD_ACCUM,"effective_batch":EFFECTIVE_BATCH,"max_seq_length":MAX_SEQ_LENGTH,"runtime_seconds":runtime,
        "train_metrics":result.metrics,"final_adapter":str(FINAL_ADAPTER),"next":"Evaluate every saved quarter checkpoint by fixed dev SC16; do not select by loss."}
REPORT_PATH=REPORT_DIR/"training_report.json";REPORT_PATH.write_text(json.dumps(report,ensure_ascii=False,indent=2,default=str),encoding="utf-8")
print("[ADAPTER]",FINAL_ADAPTER);print("[REPORT]",REPORT_PATH);print("[NEXT] SC16 checkpoint sweep required")

In [ ]:
# Final cell — Optional GPU runtime release after adapter/report saves finish.
DISCONNECT_GPU_RUNTIME = False
if DISCONNECT_GPU_RUNTIME:
    from google.colab import runtime
    runtime.unassign()
else:
    print("[RUNTIME] retained. Set DISCONNECT_GPU_RUNTIME=True to release the GPU.")